# How Long to Beat Data Merger

This notebook merges **How Long to Beat** completion time data with the IGN Steam merged games dataset. We'll fetch completion times for different play styles:

- **Main Story**: Time to complete just the main story
- **Main + Extra**: Time to complete main story plus some side content
- **Completionist**: Time to 100% complete the game
- **All Styles**: Average time across all play styles

The process includes:
1. Loading and exploring the dataset
2. Setting up the HowLongToBeat API client
3. Processing game titles for better matching
4. Fetching completion time data
5. Saving the enriched dataset

## 1. Import Required Libraries

First, let's import all the necessary libraries for data processing and API interaction.

In [2]:
# Data processing libraries
import pandas as pd
import numpy as np

# HowLongToBeat API
from howlongtobeatpy import HowLongToBeat

# Utility libraries
import time
import re
from typing import Optional, Dict, Any
from tqdm import tqdm

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## 2. Load and Explore the Dataset

Let's load our IGN Steam merged games dataset and take a look at its structure.

In [3]:
# Delete any existing input.csv
import os
if os.path.exists("input.csv"):
    os.remove("input.csv")
    print("🗑️ Removed existing input.csv")

# Copy the merged dataset to input.csv
output_path = "../1_merge_ign_steam/output.csv"

import shutil
shutil.copyfile(output_path, "input.csv")
print(f"📋 Copied {output_path} to input.csv")

# Load the merged dataset
df = pd.read_csv("input.csv")


print(f"📊 Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

# Display first few rows
df.head()

📋 Copied ../1_merge_ign_steam/output.csv to input.csv
📊 Dataset loaded successfully!
Shape: (613, 14)
Columns: ['title', 'release_date', 'genre', 'developer', 'publisher', 'required_age', 'critic_score', 'critic_score_phrase', 'user_positive', 'user_negative', 'average_playtime', 'median_playtime', 'owners', 'price']


,title,release_date,genre,developer,publisher,required_age,critic_score,critic_score_phrase,user_positive,user_negative,average_playtime,median_playtime,owners,price
0,Home: A Unique Horror Adventure,2012-09-06,Adventure,Benjamin Rivers Inc.,Benjamin Rivers Inc.,0,6.5,Okay,1069,518,21,21,350000,1.99
1,Symphony,2012-08-30,Shooter,Empty Clip Studios,Empty Clip Studios,0,7.0,Good,1159,209,321,321,350000,6.19
2,Thirty Flights of Loving,2012-08-29,Adventure,Blendo Games,Blendo Games,0,8.0,Great,934,616,0,0,150000,3.99
3,Worms Revolution,2012-10-02,Strategy,Team17 Digital Ltd,Team17 Digital Ltd,0,8.5,Great,3922,686,445,259,1500000,10.99
4,Shad'O,2012-09-28,Adventure,Okugi Studio,Okugi Sudio,0,7.0,Good,50,42,58,58,35000,3.99


In [3]:
# Check for missing values in title column
print("🔍 Data Quality Check:")
print(f"Total games: {len(df)}")
print(f"Games with valid titles: {df['title'].notna().sum()}")
print(f"Games with missing titles: {df['title'].isna().sum()}")

# Show some example titles
print("\n📝 Sample game titles:")
sample_titles = df['title'].dropna().sample(n=min(10, len(df))).tolist()
for i, title in enumerate(sample_titles, 1):
    print(f"{i}. {title}")

🔍 Data Quality Check:
Total games: 613
Games with valid titles: 613
Games with missing titles: 0

📝 Sample game titles:
1. Scribblenauts Unmasked: A DC Comics Adventure
2. Borderlands 2
3. Nancy Drew: Ransom of the Seven Ships
4. Waking Mars
5. Hotline Miami
6. Necropolis
7. The Path
8. Dragon Age: Origins (Collector's Edition)
9. Strike Suit Zero
10. Red Orchestra: Ostfront 41-45


## 3. Setup HowLongToBeat API and Helper Functions

We'll create helper functions to clean game titles and search for completion time data.

In [4]:
# Initialize HowLongToBeat client
hltb = HowLongToBeat()

def clean_game_title(title: str) -> str:
    """
    Clean the game title for better matching with HowLongToBeat database.
    
    This function removes common suffixes, edition information, and special characters
    that might interfere with the search process.
    """
    if pd.isna(title):
        return ""

    title = str(title)

    # Remove version numbers and years in parentheses
    title = re.sub(r"\([^)]*\)", "", title)

    # Remove trademark symbols
    title = re.sub(r"[™®©]", "", title)

    # Replace -- with - in title
    title = title.replace("--", "-")

    # Remove extra whitespace
    title = " ".join(title.split())

    return title.strip()

def extra_clean_game_title(cleaned_title: str) -> str:
    """
    Further clean the cleaned game title by removing anything after the last colon or dash.
    """
    # Remove anything after the last colon or dash
    cleaned_title = re.split(r"[:\-]", cleaned_title)[0].strip()
    
    # Remove extra whitespace again
    cleaned_title = " ".join(cleaned_title.split())
    
    return cleaned_title

# Test the cleaning function
test_titles = [
    "Grand Theft Auto V: Premium Edition",
    "The Witcher 3: Wild Hunt - Game of the Year Edition",
    "Cyberpunk 2077™",
    "Red Dead Redemption 2 (2018)",
    "Star Wars: The Force Unleashed -- Ultimate Sith Edition",
    "Warhammer 40,000: Dawn of War -- Soulstorm",
    "Ninja Reflex: Steamworks Edition",
]

print("🧹 Title Cleaning Examples:")
for original in test_titles:
    cleaned = clean_game_title(original)
    print(f"Original: {original}")
    print(f"Cleaned:  {cleaned}")
    print("---")

🧹 Title Cleaning Examples:
Original: Grand Theft Auto V: Premium Edition
Cleaned:  Grand Theft Auto V: Premium Edition
---
Original: The Witcher 3: Wild Hunt - Game of the Year Edition
Cleaned:  The Witcher 3: Wild Hunt - Game of the Year Edition
---
Original: Cyberpunk 2077™
Cleaned:  Cyberpunk 2077
---
Original: Red Dead Redemption 2 (2018)
Cleaned:  Red Dead Redemption 2
---
Original: Star Wars: The Force Unleashed -- Ultimate Sith Edition
Cleaned:  Star Wars: The Force Unleashed - Ultimate Sith Edition
---
Original: Warhammer 40,000: Dawn of War -- Soulstorm
Cleaned:  Warhammer 40,000: Dawn of War - Soulstorm
---
Original: Ninja Reflex: Steamworks Edition
Cleaned:  Ninja Reflex: Steamworks Edition
---


In [5]:
def search_game_hltb(title: str, max_retries: int = 3) -> Optional[Dict[str, Any]]:
    """
    Search for a game on HowLongToBeat and return the completion times.
    
    Args:
        title: Game title to search for
        max_retries: Maximum number of retry attempts
        
    Returns:
        Dictionary containing completion times or None if not found
    """
    if not title:
        return None

    cleaned_title = clean_game_title(title)

    for attempt in range(max_retries):
        try:
            # Search for the game
            results = hltb.search(cleaned_title)

            if results and len(results) > 0:
                # Get the first (most relevant) result
                game = results[0]

                return {
                    "main_story": getattr(game, "main_story", 0),
                    "main_extra": getattr(game, "main_extra", 0),
                    "completionist": getattr(game, "completionist", 0),
                    "all_styles": getattr(game, "all_styles", 0),
                }

            # If no exact match, try with extra cleaning
            extra_cleaned_title = extra_clean_game_title(cleaned_title)
            
            if extra_cleaned_title == cleaned_title:
                return None  # No further cleaning possible
            
            results = hltb.search(extra_cleaned_title)
            if results and len(results) > 0:
                game = results[0]
                return {
                    "main_story": getattr(game, "main_story", 0),
                    "main_extra": getattr(game, "main_extra", 0),
                    "completionist": getattr(game, "completionist", 0),
                    "all_styles": getattr(game, "all_styles", 0),
                }

            return None

        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2)  # Wait before retry
                continue
            else:
                return None

# Test the search function with a few games
test_games = ["The Witcher 3", "Cyberpunk 2077", "Minecraft", "Warhammer 40,000: Dawn of War -- Soulstorm",
    "Ninja Reflex: Steamworks Edition"]

print("🔍 Testing HLTB Search Function:")
for game in test_games:
    result = search_game_hltb(game)
    if result:
        print(f"✅ {game}: All Styles = {result['all_styles']} hours")
    else:
        print(f"❌ {game}: No data found")
    time.sleep(1)  # Be respectful to the API

🔍 Testing HLTB Search Function:
✅ The Witcher 3: All Styles = 103.77 hours
✅ Cyberpunk 2077: All Styles = 65.19 hours
✅ Minecraft: All Styles = 133.54 hours
✅ Warhammer 40,000: Dawn of War -- Soulstorm: All Styles = 20.82 hours
✅ Ninja Reflex: Steamworks Edition: All Styles = 3.9 hours


## 4. Process a Sample of Games (Demo)

Let's first test our process on a small sample of games to see how it works before processing the entire dataset.

In [6]:
# Create a sample dataset for testing
sample_size = 20
sample_df = df.sample(n=min(sample_size, len(df)), random_state=42).copy().reset_index(drop=True)

print(f"🎮 Processing sample of {len(sample_df)} games:")

# Initialize new columns
sample_df["main_story"] = pd.Series(dtype="float64")
sample_df["main_extra"] = pd.Series(dtype="float64")
sample_df["completionist"] = pd.Series(dtype="float64")
sample_df["all_styles"] = pd.Series(dtype="float64")

hltb_columns = ["main_story", "main_extra", "completionist", "all_styles"]

# Process each game in the sample
successful_matches = 0
progress_bar = tqdm(sample_df.iterrows(), total=len(sample_df), desc="Processing games")

for index, row in progress_bar:
    title = row["title"]
    progress_bar.set_description(f"Processing: {title[:30]}...")
    
    # Search for HLTB data
    hltb_data = search_game_hltb(title)
    
    if hltb_data:
        for col in hltb_columns:
            sample_df.at[index, col] = hltb_data[col]
        successful_matches += 1
        progress_bar.set_postfix({"Found": successful_matches})
    else:
        # Set NaN for missing data
        for col in hltb_columns:
            sample_df.at[index, col] = np.nan
    
    # Be respectful to the API
    time.sleep(1)

print(f"\n✅ Sample processing complete!")
print(f"Successfully found data for {successful_matches}/{len(sample_df)} games ({successful_matches/len(sample_df)*100:.1f}%)")

🎮 Processing sample of 20 games:


Processing: Virginia...: 100%|██████████| 20/20 [00:41<00:00,  2.07s/it, Found=20]                      


✅ Sample processing complete!
Successfully found data for 20/20 games (100.0%)


In [7]:
# Display results from the sample
print("📊 Sample Results:")
display_cols = ['title'] + hltb_columns
sample_results = sample_df[display_cols]

# Show games with data found
games_with_data = sample_results[sample_results['all_styles'].notna()]
print(f"\n🎯 Games with HLTB data found ({len(games_with_data)} games):")
display(games_with_data)

# Show some statistics
if len(games_with_data) > 0:
    print("\n📈 Sample Statistics:")
    for col in hltb_columns:
        valid_data = games_with_data[col][games_with_data[col] > 0]
        if len(valid_data) > 0:
            print(f"{col.replace('_', ' ').title()}: {valid_data.mean():.1f}h avg, {valid_data.median():.1f}h median")

📊 Sample Results:

🎯 Games with HLTB data found (20 games):


,title,main_story,main_extra,completionist,all_styles
0,Amnesia: A Machine for Pigs,4.14,5.03,5.88,4.62
1,Rise of the Argonauts,12.11,13.43,16.19,13.29
2,NecroVisioN,8.91,13.34,16.70,10.31
3,Batman: Arkham Origins Blackgate -- Deluxe Edi...,7.50,10.36,18.89,10.02
4,Furi,5.50,7.61,17.50,6.50
...,...,...,...,...,...
15,Hoard,7.73,15.93,36.46,12.00
16,Hearts of Iron IV,51.37,351.17,1070.74,263.89
17,Razor2: Hidden Skies,1.73,0.00,4.25,2.36
18,Oblitus,0.67,2.50,4.79,3.12



📈 Sample Statistics:
Main Story: 9.2h avg, 7.0h median
Main Extra: 31.1h avg, 10.3h median
Completionist: 80.1h avg, 16.4h median
All Styles: 23.2h avg, 8.5h median


## 5. Process the Complete Dataset

Now let's process the entire dataset. This will take some time as we need to be respectful to the HLTB API with rate limiting.

In [8]:
# Check if we want to process the full dataset
process_full_dataset = True  # Set to False if you want to skip this step

if process_full_dataset:
    print("🚀 Starting full dataset processing...")
    print(f"Total games to process: {len(df)}")
    print("⏱️  Estimated time: ~{:.1f} minutes (with 0.5-second delays)".format(len(df) / 60 / 2))
    
    # Initialize new columns for the full dataset
    df["main_story"] = pd.Series(dtype="float64")
    df["main_extra"] = pd.Series(dtype="float64")
    df["completionist"] = pd.Series(dtype="float64")
    df["all_styles"] = pd.Series(dtype="float64")
    
    # Check if progress file exists (for resuming interrupted processing)
    progress_file = "hltb_progress.csv"
    start_index = 0
    
    try:
        if pd.io.common.file_exists(progress_file):
            progress_df = pd.read_csv(progress_file)
            df = progress_df
            start_index = df[df["all_styles"].notna()].shape[0]
            print(f"📂 Resuming from index {start_index} (found progress file)")
    except Exception as e:
        print(f"⚠️  Could not load progress file: {e}")
    
    print(f"\n▶️  Processing {len(df) - start_index} remaining games...")
else:
    print("⏸️  Skipping full dataset processing (set process_full_dataset=True to enable)")

🚀 Starting full dataset processing...
Total games to process: 613
⏱️  Estimated time: ~5.1 minutes (with 0.5-second delays)

▶️  Processing 613 remaining games...


In [9]:
if process_full_dataset:
    # Process the full dataset
    successful_matches = df[df["all_styles"].notna()].shape[0]  # Count existing matches
    
    # Get the remaining rows to process
    remaining_rows = df.iloc[start_index:].copy()
    
    # Create progress bar for remaining rows only
    progress_bar = tqdm(
        remaining_rows.iterrows(), 
        total=len(remaining_rows), 
        desc="Processing games"
    )
    
    for index, row in progress_bar:
        title = row["title"]
        progress_bar.set_description(f"Processing: {title[:25]}...")
        
        # Search for HLTB data
        hltb_data = search_game_hltb(title)
        
        if hltb_data:
            for col in hltb_columns:
                df.at[index, col] = hltb_data[col]
            successful_matches += 1
        else:
            # Set NaN for missing data
            for col in hltb_columns:
                df.at[index, col] = np.nan
        
        progress_bar.set_postfix({
            "Found": successful_matches, 
            "Success Rate": f"{successful_matches/(index+1)*100:.1f}%"
        })
        
        # Save progress every 50 games
        if (index + 1) % 50 == 0:
            df.to_csv(progress_file, index=False)
        
        # Be respectful to the API
        time.sleep(1/2)
    
    print(f"\n🎉 Full dataset processing complete!")
    print(f"Successfully found data for {successful_matches}/{len(df)} games ({successful_matches/len(df)*100:.1f}%)")


Processing: Inside...: 100%|██████████| 613/613 [15:25<00:00,  1.51s/it, Found=609, Success Rate=99.3%]                    


🎉 Full dataset processing complete!
Successfully found data for 609/613 games (99.3%)


## 6. Analyze the Results

Let's explore the completion time data we've gathered and create some insights.

In [10]:
# Create analysis dataset (use sample_df if full dataset wasn't processed)
analysis_df = df if process_full_dataset else sample_df

print("📊 HLTB Data Analysis")
print("=" * 50)

# Basic statistics
total_games = len(analysis_df)
games_with_data = analysis_df[analysis_df['all_styles'].notna()].shape[0]
success_rate = games_with_data / total_games * 100

print(f"📈 Overall Statistics:")
print(f"  Total games: {total_games:,}")
print(f"  Games with HLTB data: {games_with_data:,}")
print(f"  Success rate: {success_rate:.1f}%")

# Analyze each completion type
print(f"\n⏱️  Completion Time Statistics:")
for col in hltb_columns:
    valid_data = analysis_df[col][(analysis_df[col] > 0) & (analysis_df[col].notna())]
    if len(valid_data) > 0:
        print(f"\n{col.replace('_', ' ').title()}:")
        print(f"  Games with data: {len(valid_data):,}")
        print(f"  Average: {valid_data.mean():.1f} hours")
        print(f"  Median: {valid_data.median():.1f} hours")
        print(f"  Min: {valid_data.min():.1f} hours")
        print(f"  Max: {valid_data.max():.1f} hours")
        print(f"  Standard deviation: {valid_data.std():.1f} hours")

📊 HLTB Data Analysis
📈 Overall Statistics:
  Total games: 613
  Games with HLTB data: 609
  Success rate: 99.3%

⏱️  Completion Time Statistics:

Main Story:
  Games with data: 591
  Average: 14.1 hours
  Median: 8.0 hours
  Min: 0.3 hours
  Max: 772.1 hours
  Standard deviation: 36.2 hours

Main Extra:
  Games with data: 563
  Average: 28.7 hours
  Median: 12.3 hours
  Min: 0.6 hours
  Max: 1132.5 hours
  Standard deviation: 72.3 hours

Completionist:
  Games with data: 557
  Average: 77.1 hours
  Median: 20.9 hours
  Min: 0.5 hours
  Max: 6371.1 hours
  Standard deviation: 332.6 hours

All Styles:
  Games with data: 601
  Average: 25.9 hours
  Median: 10.8 hours
  Min: 0.4 hours
  Max: 1270.8 hours
  Standard deviation: 72.7 hours


In [11]:
# Show top games by completion time
games_with_times = analysis_df[analysis_df['all_styles'] > 0].copy()

if len(games_with_times) > 0:
    print("🏆 Top 10 Longest Games (All Styles):")
    longest_games = games_with_times.nlargest(10, 'all_styles')[['title', 'all_styles']]
    for idx, (_, row) in enumerate(longest_games.iterrows(), 1):
        print(f"{idx:2d}. {row['title']} - {row['all_styles']:.1f} hours")
    
    print("\n⚡ Top 10 Shortest Games (All Styles):")
    shortest_games = games_with_times.nsmallest(10, 'all_styles')[['title', 'all_styles']]
    for idx, (_, row) in enumerate(shortest_games.iterrows(), 1):
        print(f"{idx:2d}. {row['title']} - {row['all_styles']:.1f} hours")

🏆 Top 10 Longest Games (All Styles):
 1. Dota 2 - 1270.8 hours
 2. Team Fortress 2 - 603.2 hours
 3. Warframe - 560.4 hours
 4. America's Army 3 - 512.3 hours
 5. Counter-Strike: Global Offensive - 414.9 hours
 6. Europa Universalis IV - 367.1 hours
 7. Hearts of Iron IV - 263.9 hours
 8. Kerbal Space Program - 244.8 hours
 9. PlanetSide 2 - 225.7 hours
10. Natural Selection 2 - 181.5 hours

⚡ Top 10 Shortest Games (All Styles):
 1. Thirty Flights of Loving - 0.4 hours
 2. Superhot - 0.5 hours
 3. Samurai Gunn - 0.5 hours
 4. Proteus - 1.1 hours
 5. Nidhogg - 1.2 hours
 6. Minecraft: Story Mode -- Episode 2: Assembly Required - 1.2 hours
 7. Critical Mass - 1.2 hours
 8. Blueberry Garden - 1.3 hours
 9. A Bird Story - 1.3 hours
10. Home: A Unique Horror Adventure - 1.4 hours


In [12]:
# Create completion time distribution
if len(games_with_times) > 0:
    print("📊 Completion Time Distribution (All Styles):")
    
    # Create bins for different game lengths
    bins = [0, 5, 10, 20, 50, 100, float('inf')]
    labels = ['0-5h', '5-10h', '10-20h', '20-50h', '50-100h', '100h+']
    
    games_with_times['time_category'] = pd.cut(games_with_times['all_styles'], bins=bins, labels=labels, right=False)
    distribution = games_with_times['time_category'].value_counts().sort_index()
    
    print("\nGame Length Categories:")
    for category, count in distribution.items():
        percentage = count / len(games_with_times) * 100
        print(f"  {category}: {count:,} games ({percentage:.1f}%)")

📊 Completion Time Distribution (All Styles):

Game Length Categories:
  0-5h: 115 games (19.1%)
  5-10h: 167 games (27.8%)
  10-20h: 152 games (25.3%)
  20-50h: 113 games (18.8%)
  50-100h: 34 games (5.7%)
  100h+: 20 games (3.3%)


In [13]:
# Fix the title formatting for "Who's That Flying!?" game
# Find and update the specific game title
mask = df['title'].str.contains("Who's That Flying", na=False)
if mask.any():
    # Update the title to have ?! instead of !?
    df.loc[mask, 'title'] = df.loc[mask, 'title'].str.replace("!?", "?!")
    updated_title = df.loc[mask, 'title'].iloc[0]
    print(f"✅ Updated title: {updated_title}")
    
    # Search for HLTB data with the corrected title
    hltb_data = search_game_hltb(updated_title)
    
    if hltb_data:
        # Update the HLTB columns for this game
        for col in hltb_columns:
            df.loc[mask, col] = hltb_data[col]
        print(f"✅ Found HLTB data: All Styles = {hltb_data['all_styles']} hours")
    else:
        print("❌ No HLTB data found for corrected title")
        
else:
    print("❌ Game 'Who's That Flying' not found")

✅ Updated title: Who's That Flying?!
✅ Found HLTB data: All Styles = 2.48 hours


In [14]:
# Show games that didn't have HLTB data
games_without_data = analysis_df[analysis_df['all_styles'].isna()].copy()

print("❌ Games WITHOUT HLTB Data:")
print("=" * 50)
print(f"Total games without data: {len(games_without_data):,}")
print(f"Percentage of dataset: {len(games_without_data)/len(analysis_df)*100:.1f}%")

if len(games_without_data) > 0:
    print(f"\n📝 List of games without HLTB data:")
    print("-" * 40)
    
    # Show all games without data
    for idx, (_, row) in enumerate(games_without_data.iterrows(), 1):
        title = row['title']
        genre = row.get('genre', 'Unknown')
        release_date = row.get('release_date', 'Unknown')
        print(f"{idx:3d}. {title} ({genre}, {release_date})")
    
    # Show some statistics about games without data
    print(f"\n📊 Analysis of games without HLTB data:")
    
    # Genre distribution
    if 'genre' in games_without_data.columns:
        genre_counts = games_without_data['genre'].value_counts().head(5)
        print(f"\nTop genres without data:")
        for genre, count in genre_counts.items():
            print(f"  {genre}: {count} games")
    
    # Release year analysis (if possible)
    if 'release_date' in games_without_data.columns:
        try:
            games_without_data['year'] = pd.to_datetime(games_without_data['release_date']).dt.year
            year_counts = games_without_data['year'].value_counts().head(5)
            print(f"\nTop release years without data:")
            for year, count in year_counts.items():
                print(f"  {year}: {count} games")
        except:
            print("\n⚠️ Could not analyze release years")
else:
    print("\n🎉 All games have HLTB data!")

❌ Games WITHOUT HLTB Data:
Total games without data: 3
Percentage of dataset: 0.5%

📝 List of games without HLTB data:
----------------------------------------
  1. E.Y.E.: Divine Cybermancy (Shooter, 2011-08-05)
  2. Penny Arcade's On the Rain-Slick Precipice of Darkness 3 (RPG, 2012-06-25)
  3. Penny Arcade's On the Rain-slick Precipice of Darkness 4 (RPG, 2013-06-26)

📊 Analysis of games without HLTB data:

Top genres without data:
  RPG: 2 games
  Shooter: 1 games

Top release years without data:
  2011: 1 games
  2012: 1 games
  2013: 1 games


In [15]:
# Remove games without HLTB data
print("🗑️ Removing games without HLTB data...")
print(f"Original dataset size: {len(df)} games")

# Filter out games where all_styles is NaN
df_cleaned = df.dropna(subset=['all_styles']).copy()

print(f"Cleaned dataset size: {len(df_cleaned)} games")
print(f"Removed: {len(df) - len(df_cleaned)} games")

# Update the main dataframe
df = df_cleaned.reset_index(drop=True)

print(f"\n✅ Dataset cleaned successfully!")
print(f"All {len(df)} games now have HLTB completion time data")

# Show the games that were removed
if len(games_without_data) > 0:
    print(f"\n📝 Removed games:")
    for idx, (_, row) in enumerate(games_without_data.iterrows(), 1):
        print(f"  {idx}. {row['title']}")

🗑️ Removing games without HLTB data...
Original dataset size: 613 games
Cleaned dataset size: 610 games
Removed: 3 games

✅ Dataset cleaned successfully!
All 610 games now have HLTB completion time data

📝 Removed games:
  1. E.Y.E.: Divine Cybermancy
  2. Penny Arcade's On the Rain-Slick Precipice of Darkness 3
  3. Penny Arcade's On the Rain-slick Precipice of Darkness 4


## 7. Save the Enhanced Dataset

Finally, let's save our enhanced dataset with the HLTB completion time data.

In [16]:
# Save the enhanced dataset
output_file = "output.csv"
final_df = df if process_full_dataset else sample_df

# Select only the columns we want in the final output
# (excluding hltb_id and hltb_name as requested)
output_columns = [col for col in final_df.columns if col not in ['hltb_id', 'hltb_name']]
final_output_df = final_df[output_columns]

# Save to CSV
final_output_df.to_csv(output_file, index=False)

print(f"💾 Enhanced dataset saved to: {output_file}")
print(f"📏 Final dataset shape: {final_output_df.shape}")
print(f"📋 Columns in final dataset: {len(final_output_df.columns)}")

# Show the final column list
print("\n📝 Final dataset columns:")
for i, col in enumerate(final_output_df.columns, 1):
    marker = "🆕" if col in hltb_columns else "📋"
    print(f"  {i:2d}. {marker} {col}")

💾 Enhanced dataset saved to: output.csv
📏 Final dataset shape: (610, 18)
📋 Columns in final dataset: 18

📝 Final dataset columns:
   1. 📋 title
   2. 📋 release_date
   3. 📋 genre
   4. 📋 developer
   5. 📋 publisher
   6. 📋 required_age
   7. 📋 critic_score
   8. 📋 critic_score_phrase
   9. 📋 user_positive
  10. 📋 user_negative
  11. 📋 average_playtime
  12. 📋 median_playtime
  13. 📋 owners
  14. 📋 price
  15. 🆕 main_story
  16. 🆕 main_extra
  17. 🆕 completionist
  18. 🆕 all_styles


In [4]:
import os

# Delete the progress file if it exists
progress_file = "hltb_progress.csv"
if os.path.exists(progress_file):
    os.remove(progress_file)
    print(f"🗑️ Deleted temporary progress file: {progress_file}")
else:
    print(f"ℹ️ No progress file found to delete")

print("✅ Cleanup complete!")

ℹ️ No progress file found to delete
✅ Cleanup complete!


In [17]:
# Final summary and preview
print("🎉 PROCESS COMPLETE!")
print("=" * 50)

# Show a preview of the enhanced data
print("\n👀 Preview of enhanced dataset:")
preview_cols = ['title', 'main_story', 'main_extra', 'completionist', 'all_styles']
preview_data = final_output_df[preview_cols].head(10)
display(preview_data)

print(f"\n📈 Summary:")
print(f"  ✅ Added 4 new columns with HLTB completion times")
print(f"  ✅ Processed {len(final_output_df):,} games total")
print(f"  ✅ Found completion data for {final_output_df['all_styles'].notna().sum():,} games")
print(f"  ✅ Success rate: {final_output_df['all_styles'].notna().sum()/len(final_output_df)*100:.1f}%")
print(f"  ✅ Dataset saved as: {output_file}")

🎉 PROCESS COMPLETE!

👀 Preview of enhanced dataset:


,title,main_story,main_extra,completionist,all_styles
0,Home: A Unique Horror Adventure,1.15,1.54,2.96,1.38
1,Symphony,8.08,12.70,20.93,9.93
2,Thirty Flights of Loving,0.31,0.55,0.70,0.41
3,Worms Revolution,18.92,32.51,49.87,26.62
4,Shad'O,6.11,10.32,21.75,10.81
5,Counter-Strike: Global Offensive,120.95,365.22,775.31,414.88
6,Hotline Miami,5.24,7.32,15.21,6.76
7,Edna & Harvey: Harvey's New Eyes,7.48,8.53,9.71,8.28
8,Doom 3: BFG Edition,9.53,15.98,37.78,13.84
9,Borderlands 2: Captain Scarlett and her Pirate...,3.96,6.69,10.48,6.30



📈 Summary:
  ✅ Added 4 new columns with HLTB completion times
  ✅ Processed 610 games total
  ✅ Found completion data for 610 games
  ✅ Success rate: 100.0%
  ✅ Dataset saved as: output.csv
